In [15]:
import pandas as pd
from src.assay_calibration.data_utils.dataset import Scoreset
from src.assay_calibration.fit_utils.fit import Fit
from argparse import Namespace
from tqdm.auto import tqdm, trange
from pathlib import Path
import pickle
pd.set_option("display.max_columns",None)

In [2]:
df = pd.read_csv("/data/projects/igvf/assay_calibration/dataframe_expanded.csv.tar.gz")

/tmp/ipykernel_2971574/1443624977.py:1: DtypeWarning: Columns (4,11,22,23,24,31,36,37,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,57,69,71,72,73,74,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/data/projects/igvf/assay_calibration/dataframe_expanded.csv.tar.gz")


In [6]:
df[df.Gene == "FKRP"].Dataset.unique()

array(['FKRP_Ma_2024'], dtype=object)

In [8]:
scoreset_params = {
    'clinvar_2018_0star': Namespace(clinvar_release='2018',
                                    min_clinvar_star=0),
    'clinvar_2018_2star': Namespace(clinvar_release='2018',
                                    min_clinvar_star=2),
    'clinvar_2025_0star': Namespace(clinvar_release='2025',
                                    min_clinvar_star=0),
    'clinvar_2025_2star': Namespace(clinvar_release='2025',
                                    min_clinvar_star=2),
}
scoreset_args = [("BRCA1_Findlay_2018",list(scoreset_params.keys())),
                 ("MSH2_Jia_2021", list(scoreset_params.keys())),
                 ("VHL_Buckley_2024", ['clinvar_2025_0star','clinvar_2025_2star']),
                 ('FKRP_Ma_2024',['clinvar_2025_0star',]),
                 ("LARGE1_Ma_2024",['clinvar_2025_0star'])]

In [11]:
scoresets = {}
for scoreset_name, scoreset_paramsets in tqdm(scoreset_args):
    scoreset_df = df[df.Dataset == scoreset_name]
    for paramset_name in tqdm(scoreset_paramsets,leave=False):
        scoreset = Scoreset(scoreset_df,**scoreset_params[paramset_name].__dict__)
        scoresets[(scoreset_name,paramset_name)] = scoreset

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
NBootstraps = 1000
save_rt = Path("/data/projects/igvf/assay_calibration/benchmarking/fits")
jobs_save_rt = Path("/data/projects/igvf/assay_calibration/benchmarking/jobs")
total_jobs = 0
for (scoreset_name,paramset_name), scoreset in tqdm(list(scoresets.items())):
    fit = Fit(scoreset)
    uid = "_".join((scoreset_name,paramset_name))
    jobs_save_dir = jobs_save_rt / uid
    jobs_save_dir.mkdir(exist_ok=True,parents=True)
    for bootstrap_seed in trange(NBootstraps,leave=False):
        scoreset_jobs = fit.generate_fit_jobs([2,3],
                                              save_rt / "_".join((scoreset_name,paramset_name)),
                                              bootstrap_seed=bootstrap_seed)
        for jobNum,job in enumerate(scoreset_jobs):
            with open(jobs_save_dir / f"job_{job['job_id']}.pkl",'wb') as f:
                pickle.dump(job,f)
            total_jobs+=1
print(f"Wrote {total_jobs:,d} jobs to {jobs_save_rt}")

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

...

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Wrote 2,400,000 jobs to /data/projects/igvf/assay_calibration/benchmarking/jobs


In [17]:
for scoreset_id, scoreset in scoresets.items():
    print(scoreset_id)
    print(scoreset)

('BRCA1_Findlay_2018', 'clinvar_2018_0star')
BRCA1_Findlay_2018: 3893 total variants
	Pathogenic/Likely Pathogenic: 268 variants
	Benign/Likely Benign: 90 variants
	population: 599 variants
	Synonymous: 544 variants

('BRCA1_Findlay_2018', 'clinvar_2018_2star')
BRCA1_Findlay_2018: 3893 total variants
	Pathogenic/Likely Pathogenic: 176 variants
	Benign/Likely Benign: 39 variants
	population: 599 variants
	Synonymous: 544 variants

('BRCA1_Findlay_2018', 'clinvar_2025_0star')
BRCA1_Findlay_2018: 3893 total variants
	Pathogenic/Likely Pathogenic: 419 variants
	Benign/Likely Benign: 271 variants
	population: 599 variants
	Synonymous: 544 variants

('BRCA1_Findlay_2018', 'clinvar_2025_2star')
BRCA1_Findlay_2018: 3893 total variants
	Pathogenic/Likely Pathogenic: 305 variants
	Benign/Likely Benign: 108 variants
	population: 599 variants
	Synonymous: 544 variants

('MSH2_Jia_2021', 'clinvar_2018_0star')
MSH2_Jia_2021: 51846 total variants
	Pathogenic/Likely Pathogenic: 50 variants
	Benign/Lik

In [ ]:
uvals = {}
for column in ['clinvar_sig_2025', 'clinvar_star_2025', 'clinvar_sig_2018', 'clinvar_star_2018']:
   uvals[column] = set(df[column].unique().tolist())

In [ ]:
uvals.keys()

In [ ]:
uvals['clinvar_sig_2018'] == uvals['clinvar_sig_2025']

In [ ]:
uvals['clinvar_sig_2018'] - uvals['clinvar_sig_2025']

In [ ]:
uvals['clinvar_sig_2025'] - uvals['clinvar_sig_2018']

In [ ]:
uvals['clinvar_sig_2018'].intersection(uvals['clinvar_sig_2025'])

In [ ]:
uvals['clinvar_star_2018']

In [ ]:
uvals['clinvar_star_2025']

In [ ]:
getattr(df.iloc[0],'clinvar_sig_2025')